In [3]:
from recbole.quick_start import run_recbole
import torch

In [9]:
!pip install thop

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

fastai 2.7.14 requires torch<2.3,>=1.10, but you have torch 2.3.1 which is incompatible.

     ------------------------------------ 228.5/228.5 MB 833.5 kB/s eta 0:00:00


In [23]:
# Specify the model configuration
config_dict = {
    'model': 'BERT4Rec',
    'dataset': 'ml-1m',  # Replace with your dataset
    'train_batch_size': 512,
    'eval_batch_size': 512,
    'max_seq_length': 50,
    'epochs': 1,
    'learning_rate': 0.001,
    'embedding_size': 64,
    'hidden_size': 64,
    'num_layers': 2,
    'num_heads': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'loss_type': 'CE',  # Cross-Entropy loss
    'train_neg_sample_args': None,  # Ensure negative sampling is disabled for CE loss
    'eval_neg_sample_args': None  # Same for evaluation (can also be set to a different value if needed)
}

# Run the model with the given configuration
result = run_recbole(model='BERT4Rec', config_dict=config_dict)

# Extract the model for further use
model = result['model']

31 Aug 13:10    INFO  ['D:\\Anaconda\\lib\\site-packages\\ipykernel_launcher.py', '-f', 'C:\\Users\\Hooman\\AppData\\Roaming\\jupyter\\runtime\\kernel-8b16ab1d-d690-445b-95e1-14ea9ec9ee26.json']
31 Aug 13:10    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = dataset/ml-1m
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 1
train_batch_size = 512
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = Tru

ValueError: [timestamp] is not exist in interaction [The batch_size of interaction: 1000209
    user_id, torch.Size([1000209]), cpu, torch.int64
    item_id, torch.Size([1000209]), cpu, torch.int64

].

In [3]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.sequential_recommender import BERT4Rec
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

# Configuration directly in the script
config_dict = {
    'model': 'BERT4Rec',
    'dataset': 'ml-1m',
    'epochs': 5,
    'learning_rate': 0.001,
    'train_batch_size': 512,
    'eval_batch_size': 256,
    'embedding_size': 64,
    'max_seq_length': 50,
    'hidden_size': 64,
    'num_hidden_layers': 2,
    'num_attention_heads': 2,
    'dropout_prob': 0.2,
    'seed': 42,  # Ensure reproducibility
    'reproducibility': True,
    'device': 'cuda',  # Use 'cpu' if you don't have a GPU
    'train_neg_sample_args': None,  # Disable negative sampling
    'eval_neg_sample_args': None,   # Disable negative sampling
    'loss_type': 'CE',  # Cross-Entropy Loss
    'load_col': {
    'inter': ['user_id', 'item_id', 'timestamp'],  # Ensure timestamp is loaded
    'user': ['user_id'],
    'item': ['item_id']
    },
    'eval_args': {
        'split': {'RS': [0.8, 0.1, 0.1]},  # Adjust to your preferred splitting strategy
        'mode': 'full',
        'order': 'TO',  # Time Order is required for sequential recommendation
        'topk': [5, 10]
    }
}

# Load the configuration
config = Config(config_dict=config_dict)

# Initialize random seed
init_seed(config['seed'], config['reproducibility'])

# Logger setup
init_logger(config)

# Load the dataset
dataset = create_dataset(config)

# Data preparation
train_data, valid_data, test_data = data_preparation(config, dataset)

# Initialize the model
model = BERT4Rec(config, train_data.dataset).to(config['device'])

# Initialize the trainer
trainer = Trainer(config, model)

# Train the model
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

# Test the model
test_result = trainer.evaluate(test_data)
print(f'Test result: {test_result}')

D:\Anaconda\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
D:\Anaconda\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, w

Test result: OrderedDict([('recall@10', 0.1543), ('mrr@10', 0.053), ('ndcg@10', 0.0763), ('hit@10', 0.1543), ('precision@10', 0.0154)])


In [ ]:
Test result: OrderedDict([('recall@10', 0.1072), ('mrr@10', 0.0353), ('ndcg@10', 0.0519), ('hit@10', 0.1072), ('precision@10', 0.0107)])
Test result: OrderedDict([('recall@10', 0.1543), ('mrr@10', 0.053), ('ndcg@10', 0.0763), ('hit@10', 0.1543), ('precision@10', 0.0154)])

In [10]:
from recbole.quick_start import run_recbole

parameter_dict = {
   'train_neg_sample_args': None,
}
run_recbole(model='BERT4Rec', dataset='ml-100k', config_dict=parameter_dict)

31 Aug 11:10    INFO  ['D:\\Anaconda\\lib\\site-packages\\ipykernel_launcher.py', '-f', 'C:\\Users\\Hooman\\AppData\\Roaming\\jupyter\\runtime\\kernel-8b16ab1d-d690-445b-95e1-14ea9ec9ee26.json']
31 Aug 11:10    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = D:\Anaconda\Lib\site-packages\recbole\config\../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user'

KeyboardInterrupt: 